# Generate Nimbus Scenario Data (Fabric-native)

Runs the same causally-correct "Toronto West 5G Home Internet degradation" golden scenario used by the local CLI
(`src/data_generator/generate.py` + `validate_scenario.py`), directly inside a Fabric Spark notebook.

**Why this notebook exists:** the local CLI requires a dev machine with Python/`uv` set up, which not every
organization allows. This notebook reuses the exact same generator and validator source files -- no logic is
duplicated -- so the output is guaranteed to be identical (same 18 tables, same 21/21 acceptance checks) whether
you run it locally or from Fabric.

**Prerequisites**
1. This notebook is attached to the `{prefix}_bronze` Lakehouse as its default lakehouse.
2. The generator source has been uploaded to `Files/src/data_generator/` in that lakehouse. This happens
   automatically when you run `deployment/Create-FabricAssets.ps1` (unless `-SkipGeneratorSourceUpload` is
   passed). If it's missing, upload `generate.py`, `validate_scenario.py`, and
   `config/scenario_config.yaml` manually into that Files path, or re-run the deployment script.

**What this notebook does**
1. Installs `faker`/`pyyaml` into the session (does not touch your local machine).
2. Loads `generate.py` and `validate_scenario.py` from Lakehouse Files via `notebookutils`, and imports
   them as regular Python modules.
3. Calls `build_datasets(config)` to generate all 18 tables in memory.
4. Writes each table as a managed Delta table in the attached Bronze lakehouse, and optionally also as CSVs
   under `Files/landing/raw/` for parity with the local-CLI runbook.
5. Runs `run_validation()` against the freshly written CSVs and prints a PASS/FAIL summary (target: 21/21).

In [ ]:
%pip install faker pyyaml -q

In [ ]:
# Parameters cell -- override these when running via a Fabric pipeline/job, or edit inline for ad-hoc runs.

# Name of the attached Bronze lakehouse (used only for display; the notebook writes to whichever
# lakehouse is attached as default).
bronze_lakehouse_name = "nimbus_iq_bronze"

# Where the generator source was uploaded (relative to the attached lakehouse's Files root).
generator_source_dir = "src/data_generator"

# Table naming prefix for the Delta tables written by this notebook.
table_prefix = "raw_"

# What to write: "delta", "csv", or "both".
output_mode = "both"

# Optional overrides applied on top of config/scenario_config.yaml before generation.
# Example: {"seed": 20260616, "scale": {"telemetry_events": 50000}}
config_overrides: dict = {}

In [ ]:
import importlib.util
import sys
from pathlib import Path

# notebookutils is injected by the Fabric runtime; it is not available outside Fabric.
mount_point = notebookutils.fs.getMountPath("/")  # noqa: F821
lakehouse_files_root = Path(mount_point) / "Files"
generator_dir = lakehouse_files_root / generator_source_dir

generate_py = generator_dir / "generate.py"
validate_py = generator_dir / "validate_scenario.py"
config_yaml = generator_dir / "config" / "scenario_config.yaml"

missing = [p for p in (generate_py, validate_py, config_yaml) if not p.exists()]
if missing:
    missing_list = "\n".join(f"  - {p}" for p in missing)
    raise FileNotFoundError(
        "Generator source files not found in Lakehouse Files:\n"
        f"{missing_list}\n\n"
        "Run deployment/Create-FabricAssets.ps1 (without -SkipGeneratorSourceUpload) to upload them, "
        f"or manually copy src/data_generator/* into Files/{generator_source_dir}/ in this lakehouse."
    )


def _import_from_path(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


generate = _import_from_path("nimbus_generate", generate_py)
validate_scenario = _import_from_path("nimbus_validate_scenario", validate_py)

print(f"Loaded generator from {generate_py}")
print(f"Loaded validator from {validate_py}")

In [ ]:
def _deep_merge(base: dict, overrides: dict) -> dict:
    merged = dict(base)
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _deep_merge(merged[key], value)
        else:
            merged[key] = value
    return merged


config = generate.load_config(config_yaml)
if config_overrides:
    config = _deep_merge(config, config_overrides)

print(f"scenario_id: {config['scenario_id']}")
print(f"seed: {config['seed']}")
print(f"scale: {config['scale']}")

In [ ]:
# Run the exact same generation pipeline used by the local CLI -- no logic duplicated here.
datasets = generate.build_datasets(config)

for file_name, rows in datasets.items():
    print(f"{file_name}: {len(rows)} rows")

In [ ]:
# Materialize the generated rows as Delta tables and/or landing-zone CSVs, per `output_mode`.
import csv
import io

csv_landing_dir = lakehouse_files_root / "landing" / "raw"

if output_mode in ("delta", "both"):
    for file_name, rows in datasets.items():
        if not rows:
            print(f"Skipping empty dataset: {file_name}")
            continue
        table_name = f"{table_prefix}{file_name.removesuffix('.csv')}"
        df = spark.createDataFrame(rows)  # noqa: F821 (spark is injected by the Fabric runtime)
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Wrote Delta table: {table_name} ({len(rows)} rows)")

if output_mode in ("csv", "both"):
    csv_landing_dir.mkdir(parents=True, exist_ok=True)
    for file_name, rows in datasets.items():
        out_path = csv_landing_dir / file_name
        if not rows:
            out_path.write_text("", encoding="utf-8")
            continue
        with out_path.open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print(f"Wrote CSV: {out_path}")

In [ ]:
# Validate causal integrity (target: 21/21 checks passed). Requires the CSV landing files above,
# so run this only when output_mode is "csv" or "both".
if output_mode in ("csv", "both"):
    exit_code = validate_scenario.run_validation(csv_landing_dir)
    if exit_code == 0:
        print("\nAll acceptance checks passed.")
    else:
        raise RuntimeError("Validation failed -- see check output above for details.")
else:
    print("output_mode='delta': skipping CSV-based validation. Set output_mode to 'csv' or 'both' to validate.")